# Lightning Baseline Runner: U-Net / DeepLabV3+ — 17-class segmentation

Benchmarks **U-Net** and **DeepLabV3+** (from `segmentation_models_pytorch`, ImageNet-pretrained
encoders) against your SwinV2+UPerNet method on the **17-class segmentation** task — using the **same folds,
clean-original validation, focal loss, mIoU metric, and cross-fold-averaged W&B logging**.

**Switch architecture** by changing `ARCH` in the config cell: `'unet'` or `'deeplabv3plus'`.
Two notebooks (this + the sibling task) therefore cover all 4 benchmark runs.

- Pretraining→finetuning: the encoder is ImageNet-pretrained; we finetune on carbonate. (Your
  method uses domain-specific SSL — that asymmetry is exactly what the benchmark measures.)
- Same `augment_aware_kfold_indices` split, so validation = the same clean originals as your runs.
- W&B: `--wandb_mean_only` → one averaged curve per metric (+ composite per-class IoU chart).

### One-time Studio setup (terminal): rclone remote `gdrive`, cloned repo.


## 0. Pull latest code

In [ ]:
import os
REPO_ROOT = os.path.expanduser('~/Payne_Lab_Carbon_Thin_Segmentation')
os.chdir(REPO_ROOT); os.environ['REPO_ROOT'] = REPO_ROOT
!git pull origin main
print('Repo:', REPO_ROOT)

## 1. GPU check

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (open a GPU Studio)')

## 2. Install dependencies

In [ ]:
!pip -q install --upgrade segmentation-models-pytorch transformers tqdm wandb

## 2b. (Optional) Weights & Biases login
Skip to disable W&B (also drop the `--wandb_*` flags below).

In [ ]:
import os
os.environ.setdefault('WANDB_PROJECT', 'payne-carbonate-segmentation')
import wandb; wandb.login()

## 3. Verify rclone

In [ ]:
import subprocess
res = subprocess.run(['rclone','lsd','gdrive:'], capture_output=True, text=True)
if res.returncode != 0: raise RuntimeError('rclone not configured:\n'+res.stderr)
print('rclone OK')

## 4. Config — architecture, task, paths

Set **`ARCH`** to `'unet'` or `'deeplabv3plus'`. Uses the no-augmentation dataset (labelled images_PS/ALL_LABELS)
by default; see the markdown after the finetune cell to switch to the augmented dataset.

In [ ]:
import os
from pathlib import Path

ARCH  = 'unet'              # <-- 'unet' or 'deeplabv3plus'
TASK  = 'multiclass'            # fixed for this notebook

DRIVE_ROOT     = 'Petrographic images_ML work'
LABELLED_DRIVE = DRIVE_ROOT + '/labelled images_PS/ALL_LABELS'
OUT_DRIVE      = DRIVE_ROOT + '/model_outputs_lightning'

LOCAL_ROOT = Path('/teamspace/studios/this_studio/petro_data')
IMG_LOCAL  = LOCAL_ROOT / 'labeled' / 'img'
MASK_LOCAL = LOCAL_ROOT / 'labeled' / 'masks_machine'
OUT_ROOT   = LOCAL_ROOT / 'model_outputs'
for d in (IMG_LOCAL, MASK_LOCAL, OUT_ROOT): d.mkdir(parents=True, exist_ok=True)

IMG_DIR  = str(IMG_LOCAL); MASK_DIR = str(MASK_LOCAL)
OUT_DIR  = str(OUT_ROOT / f'baseline_{ARCH}_17class_3fold')
RUN_NAME = f'baseline_{ARCH}_17class'
os.environ.update(REPO_ROOT=os.environ['REPO_ROOT'], ARCH=ARCH, TASK=TASK,
                  IMG_DIR=IMG_DIR, MASK_DIR=MASK_DIR, OUT_DIR=OUT_DIR, RUN_NAME=RUN_NAME)
os.environ['WANDB_DIR'] = OUT_DIR; os.makedirs(OUT_DIR, exist_ok=True)
print('arch   :', ARCH, '| task:', TASK)
print('labeled: gdrive:' + LABELLED_DRIVE)
print('output :', OUT_DIR)

## 5. Sync labeled data from Drive

In [ ]:
import subprocess
def rclone_copy(remote_rel, local):
    subprocess.run(['rclone','copy','gdrive:'+remote_rel,str(local),
                    '--progress','--transfers=8','--checkers=16'], check=True)
rclone_copy(LABELLED_DRIVE + '/img', IMG_LOCAL)
rclone_copy(LABELLED_DRIVE + '/masks_machine', MASK_LOCAL)
print('images:', len(list(IMG_LOCAL.glob('*'))), '| masks:', len(list(MASK_LOCAL.glob('*'))))

## 6. Smoke test (dataloader only)

In [ ]:
import os
os.chdir(REPO_ROOT)
!python -u code/model_training_pipeline/baseline_seg_pipeline_221.py \
  --task "$TASK" --arch "$ARCH" --img_dir "$IMG_DIR" --mask_dir "$MASK_DIR" --ignore_artifacts --no_train

## 7. Finetune — 3-fold stratified CV, averaged W&B logging

Writes per-fold `best_<arch>_multiclass.pth` + `val_metrics.csv`, `cv_summary.json`, and one
averaged W&B run `{RUN_NAME}_cvmean`. Change `ARCH` above and re-run cells 4→7 for the other
architecture. (For long runs, copy this command into a `tmux` session instead.)

In [ ]:
import os
os.chdir(REPO_ROOT)
!python -u code/model_training_pipeline/baseline_seg_pipeline_221.py \
  --task "$TASK" --arch "$ARCH" --encoder resnet50 \
  --img_dir "$IMG_DIR" --mask_dir "$MASK_DIR" \
  --n_folds 3 --cv_strategy stratified \
  --ignore_artifacts \\
  --epochs 50 --batch_size 2 --crop 512 --num_workers 4 --lr 3e-4 \
  --warmup_epochs 5 --scheduler cosine \
  --loss_type focal --focal_gamma 2.0 \
  --output_dir "$OUT_DIR" \
  --wandb_mean_only \
  --wandb_project payne-carbonate-segmentation \
  --wandb_run_name "$RUN_NAME" \
  2>&1 | tee "$OUT_DIR/run.log"

### (Optional) benchmark on the AUGMENTED dataset instead
To match your with-augmentation runs, in cell 4 set `LABELLED_DRIVE = DRIVE_ROOT + '/augmented_and_og_labels'`
and a distinct local sub-folder (e.g. `labeled_aug`), then add these flags to the finetune command:
`--group_by_stem --num_augmentations_per_img 5`  (or 10). Validation stays clean originals.

## 8. Sync outputs to Drive

In [ ]:
import subprocess
subprocess.run(['rclone','copy', OUT_DIR, 'gdrive:'+OUT_DRIVE+'/'+os.path.basename(OUT_DIR),
                '--exclude','wandb/**','--progress','--transfers=8','--checkers=16'], check=True)
print('Synced', OUT_DIR)

## 9. Confusion matrix + per-class IoU on a chosen fold
Reproduces the exact validation split, loads a fold's baseline checkpoint, computes the
18-class confusion matrix.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np, torch
from torch.utils.data import DataLoader, Subset
from torchvision.transforms.v2 import CenterCrop, Compose

PIPE = Path(REPO_ROOT) / 'code' / 'model_training_pipeline'
if str(PIPE) not in sys.path: sys.path.insert(0, str(PIPE))
from baseline_seg_pipeline_221 import build_baseline_model, make_dataset, evaluate
from swin_training_pipeline_221 import (
    NUM_CLASSES, IGNORE_INDEX, CLASS_NAMES, artifact_ignore_ids, parse_merge_map,
    build_class_presence_matrix, stratified_kfold_indices, kfold_train_val_indices,
    augment_aware_kfold_indices, confusion_matrix as cm_fn, miou_from_confusion,
)
from swin_binary_segmentation_221 import ARTIFACT_CLASS_IDS, BINARY_CLASS_NAMES, NUM_BINARY_CLASSES

# --- MUST match the finetune run ---
ARCH='unet'; FOLD=0; N_FOLDS=3; CV_STRATEGY='stratified'; SEED=1337; CROP=512
GROUP_BY_STEM=False; GROUP_PATTERN=r'_aug\d+$'; NUM_AUGS=None
MERGE_MAP={}                      # e.g. {12:1} if you trained with --merge_class_map 12:1
TASK='multiclass'
OUT_DIR=Path('/teamspace/studios/this_studio/petro_data/model_outputs')/f'baseline_{ARCH}_17class_3fold'
CKPT=OUT_DIR/f'fold_{FOLD}'/f'best_{ARCH}_{TASK}.pth'
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if TASK=='binary':
    num_classes, class_names, ignore_ids = NUM_BINARY_CLASSES, list(BINARY_CLASS_NAMES), ARTIFACT_CLASS_IDS
    p_ignore, mmap = ARTIFACT_CLASS_IDS, None
else:
    num_classes, class_names = NUM_CLASSES, list(CLASS_NAMES)
    ignore_ids = artifact_ignore_ids(type('A',(),{'ignore_artifacts':True,'ignore_scale_bar':False})())
    p_ignore, mmap = ignore_ids, (MERGE_MAP or None)

probe = make_dataset(TASK, '.', IMG_DIR, MASK_DIR, None, ignore_ids, MERGE_MAP, strict=True, print_pair_count=False)
n=len(probe)
presence = build_class_presence_matrix(probe.pairs, NUM_CLASSES, IGNORE_INDEX, p_ignore, mmap) if CV_STRATEGY=='stratified' else None
if GROUP_BY_STEM:
    splits = augment_aware_kfold_indices(probe.pairs, GROUP_PATTERN, presence, N_FOLDS, SEED, max_augs_per_source=NUM_AUGS)
elif CV_STRATEGY=='stratified':
    splits = stratified_kfold_indices(presence, N_FOLDS, SEED)
else:
    splits = kfold_train_val_indices(n, N_FOLDS, SEED)
val_idx = list(np.asarray(splits[FOLD][1]).tolist())

val_full = make_dataset(TASK, '.', IMG_DIR, MASK_DIR, Compose([CenterCrop((CROP,CROP))]), ignore_ids, MERGE_MAP, strict=False, print_pair_count=False)
val_loader = DataLoader(Subset(val_full, val_idx), batch_size=1, shuffle=False, num_workers=0, pin_memory=torch.cuda.is_available())

model = build_baseline_model(ARCH, 'resnet50', num_classes).to(device)
model.load_state_dict(torch.load(str(CKPT), map_location=device, weights_only=False)['model_state']); model.eval()
va_loss, va_acc, va_miou, per_iou = evaluate(model, val_loader, device, num_classes, None, 'focal', 2.0)
print(f'Fold {FOLD} | val_loss {va_loss:.4f} | pixel_acc {va_acc:.3f} | mIoU {va_miou:.3f}')
print(f"\n{'class':<20}{'IoU':>8}")
for i, nm in enumerate(class_names[:num_classes]):
    v = per_iou[i].item()
    print(f'{nm:<20}' + (f'{v:>8.3f}' if v==v else '     nan'))